[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/04_shortest_paths_algorithms/exercises.ipynb)

# Exercises — Topic 04: Shortest Paths Algorithms

20 fully solved problems in 4 levels: Concept Check (4), Foundation (6), Applications in AI/ML & Physics (6), Challenge (4).

## Level 0 — Concept Check

### Problem L0.1: Which Algorithm?

For each situation, name the cheapest correct algorithm and its running time: (a) a social network with $10^6$ vertices and unit edge costs, single source; (b) a road network with positive travel times, single source; (c) a currency graph with logarithmic weights of both signs, single source; (d) a dense graph on $300$ vertices where *all* pairwise distances are needed.

**Solution**

(a) Unit costs make hop count the objective: **BFS**, $O(n + m)$. No priority queue is needed because the FIFO order already visits vertices in nondecreasing distance.

(b) Positive weights: **Dijkstra with a binary heap**, $O((n+m)\log n)$. Greedy settling is licensed by $w \ge 0$.

(c) Mixed signs: **Bellman–Ford**, $O(nm)$ — and it additionally reports whether a negative cycle (an arbitrage loop) is reachable, which Dijkstra cannot.

(d) All pairs on a dense graph ($m \approx n^2 = 9 \times 10^4$): **Floyd–Warshall**, $\Theta(n^3) = 2.7 \times 10^7$ operations on a contiguous matrix — simpler and faster in practice than $n$ Dijkstra runs at $O(nm\log n)$.

$$
\boxed{\text{(a) BFS } O(n+m); \ \text{(b) Dijkstra } O((n+m)\log n); \ \text{(c) Bellman–Ford } O(nm); \ \text{(d) Floyd–Warshall } \Theta(n^3)}
$$

*Key takeaway:* Algorithm choice is dictated by the weight assumption first (unit / nonnegative / arbitrary) and by the query pattern second (single-source / all-pairs).

### Problem L0.2: Negative Edge or Negative Cycle?

For each digraph on $\lbrace s, a, b \rbrace$, state whether $\delta(s, b)$ is well defined and compute it if so. (a) $s \to a$ of weight $2$, $a \to b$ of weight $-5$. (b) $s \to a$ of weight $2$, $a \to b$ of weight $1$, $b \to a$ of weight $-3$.

**Solution**

(a) The graph is acyclic, so every walk is a path and the minimum is attained:

$$
\delta(s,b) = 2 + (-5) = -3
$$

A negative *edge* is harmless; only the greedy order of Dijkstra objects, not the problem itself.

(b) The cycle $a \to b \to a$ has weight $1 + (-3) = -2 \lt 0$ and is reachable from $s$. Traversing it $k$ times gives an $s$–$b$ walk of weight $2 + 1 - 2k \to -\infty$, so no shortest walk exists:

$$
\delta(s,b) = -\infty
$$

Bellman–Ford would flag this by relaxing an edge on pass $n$.

$$
\boxed{\text{(a) } \delta(s,b) = -3 \text{ (well defined)}; \quad \text{(b) } \delta(s,b) = -\infty \text{ (negative cycle)}}
$$

*Key takeaway:* Negative edges are an algorithmic inconvenience; reachable negative cycles are a failure of the problem statement itself.

### Problem L0.3: Why Exactly $n-1$ Passes?

Bellman–Ford performs $n-1$ relaxation passes and then one extra pass. Explain both counts from first principles.

**Solution**

**Why $n-1$ suffices.** If no negative cycle is reachable, every optimal $s$–$v$ walk can be shortened to a *simple* path (excising any repeated vertex removes a cycle of weight $\ge 0$). A simple path on $n$ vertices has at most $n-1$ edges. Since pass $k$ makes all distances realizable with at most $k$ edges exact, $k = n-1$ makes all of them exact.

**Why one extra pass.** After $n-1$ passes the labels satisfy $d(v) \le d(u) + w(u,v)$ for every edge *precisely when* the answer is finite. So a successful relaxation on pass $n$ is a certificate that some reachable cycle has negative weight — summing the violated inequalities around the cycle cancels all labels and leaves $w(C) \lt 0$.

$$
\boxed{n-1 \text{ passes} = \text{maximum hop count of a simple path}; \quad \text{pass } n = \text{negative-cycle detector}}
$$

*Key takeaway:* The pass count tracks *hops*, never weights — which is why Bellman–Ford's complexity is independent of how large or small the numbers are.

### Problem L0.4: Admissible Versus Consistent

A heuristic satisfies $h(t) = 0$, $h(u) = 6$, $h(v) = 2$ and the edge $(u,v)$ has weight $w(u,v) = 1$. Is $h$ consistent? Could it still be admissible? What does each property buy you?

**Solution**

**Consistency** demands $h(u) \le w(u,v) + h(v)$ for every edge. Here

$$
h(u) = 6 \quad \text{versus} \quad w(u,v) + h(v) = 1 + 2 = 3
$$

so $6 \le 3$ fails: $h$ is **not consistent**. Equivalently the reduced cost $\hat{w}(u,v) = w(u,v) + h(v) - h(u) = 1 + 2 - 6 = -3 \lt 0$, so the A$^{\ast}$-as-Dijkstra reduction breaks.

It can still be **admissible**: admissibility only requires $h(u) \le \delta(u,t)$ pointwise, and nothing above contradicts, e.g., $\delta(u,t) = 7$ and $\delta(v,t) = 2$ — except that $\delta(u,t) \le w(u,v) + \delta(v,t) = 3$ by the triangle inequality, so in fact $h(u) = 6 \gt 3 \ge \delta(u,t)$ and this particular $h$ is **inadmissible too**. The inequality chain shows consistency is the stronger property.

- Admissibility $\Rightarrow$ A$^{\ast}$ returns an optimal path *if* closed nodes may be reopened.
- Consistency $\Rightarrow$ reduced costs are nonnegative, so A$^{\ast}$ is literally Dijkstra and never reopens a node.

$$
\boxed{\text{consistent} \Rightarrow \text{admissible}, \text{ not conversely}; \ \text{here } h \text{ is neither}}
$$

*Key takeaway:* Consistency is a *local* edge-by-edge condition and is what you should test in code; admissibility is a global condition that is harder to certify.

## Level 1 — Foundation

### Problem L1.1: Dijkstra Step by Step

Run Dijkstra from $s$ on the digraph with arcs $s \to a$ of weight $4$, $s \to b$ of weight $1$, $b \to a$ of weight $2$, $b \to c$ of weight $5$, $a \to c$ of weight $1$, $a \to d$ of weight $6$, $c \to d$ of weight $3$. Give the extraction order, all final distances, and the shortest-path tree.

**Solution**

Extract the minimum-key unsettled vertex and relax its outgoing arcs.

| Step | Extracted | $d$ at extraction | Relaxations performed | Labels afterwards |
|---|---|---|---|---|
| 1 | $s$ | $0$ | $a \leftarrow 4$, $b \leftarrow 1$ | $a{:}4$, $b{:}1$ |
| 2 | $b$ | $1$ | $a \leftarrow 1+2 = 3$ (improves 4), $c \leftarrow 6$ | $a{:}3$, $c{:}6$ |
| 3 | $a$ | $3$ | $c \leftarrow 3+1 = 4$ (improves 6), $d \leftarrow 9$ | $c{:}4$, $d{:}9$ |
| 4 | $c$ | $4$ | $d \leftarrow 4+3 = 7$ (improves 9) | $d{:}7$ |
| 5 | $d$ | $7$ | none | final |

Extraction order $s, b, a, c, d$ is nondecreasing in $d$, as Theorem D guarantees. Final distances

$$
\delta(s,s) = 0, \quad \delta(s,b) = 1, \quad \delta(s,a) = 3, \quad \delta(s,c) = 4, \quad \delta(s,d) = 7
$$

with predecessors $\pi(b) = s$, $\pi(a) = b$, $\pi(c) = a$, $\pi(d) = c$: the shortest-path tree is the single path $s \to b \to a \to c \to d$, and indeed $1 + 2 + 1 + 3 = 7$.

$$
\boxed{(0, 3, 1, 4, 7) \text{ for } (s,a,b,c,d); \ \text{SPT} = s \to b \to a \to c \to d}
$$

*Key takeaway:* The direct arc $s \to a$ of weight $4$ is never used — greedy settling by *distance*, not by *arc weight*, is what makes Dijkstra correct.

### Problem L1.2: Bellman–Ford Pass by Pass

Run Bellman–Ford from $s$ on the digraph with arcs $s \to a$ of weight $4$, $s \to b$ of weight $5$, $a \to c$ of weight $3$, $b \to a$ of weight $-3$, $b \to c$ of weight $6$, relaxing edges in the listed order. Show every pass and confirm no negative cycle exists.

**Solution**

Initialize $d(s) = 0$ and $d(a) = d(b) = d(c) = +\infty$.

**Pass 1** (order $sa, sb, ac, ba, bc$):

| Edge | Test | Result |
|---|---|---|
| $s \to a$ | $0 + 4 \lt \infty$ | $d(a) = 4$ |
| $s \to b$ | $0 + 5 \lt \infty$ | $d(b) = 5$ |
| $a \to c$ | $4 + 3 \lt \infty$ | $d(c) = 7$ |
| $b \to a$ | $5 - 3 = 2 \lt 4$ | $d(a) = 2$ |
| $b \to c$ | $5 + 6 = 11$, not $\lt 7$ | unchanged |

State: $(0, 2, 5, 7)$ for $(s,a,b,c)$.

**Pass 2**: only $a \to c$ fires, $2 + 3 = 5 \lt 7$, so $d(c) = 5$. State $(0, 2, 5, 5)$.

**Pass 3**: no edge relaxes — the labels already satisfy $d(v) \le d(u) + w(u,v)$ everywhere. Since $n - 1 = 3$ passes were available and convergence happened by pass 2, the answer is final.

**Detection pass** ($n = 4$): again nothing relaxes, so **no reachable negative cycle** — consistent with the graph, whose only negative arc $b \to a$ lies on no cycle.

$$
\boxed{\delta(s, \cdot) = (0, 2, 5, 5) \text{ for } (s,a,b,c); \ \text{no negative cycle}}
$$

*Key takeaway:* Edge ordering changes *when* labels converge but never the final answer; the optimal route $s \to b \to a \to c$ costs $5 - 3 + 3 = 5$ and uses the negative arc.

### Problem L1.3: Floyd–Warshall on Four Vertices

Compute all-pairs distances for the digraph with arcs $1 \to 2$ of weight $3$, $1 \to 4$ of weight $7$, $2 \to 1$ of weight $8$, $2 \to 3$ of weight $2$, $3 \to 1$ of weight $5$, $3 \to 4$ of weight $1$, $4 \to 1$ of weight $2$. Show the matrix after each $k$.

**Solution**

$D^{(0)}$ has $0$ on the diagonal, arc weights where present, $\infty$ elsewhere:

$$
D^{(0)} = \begin{pmatrix} 0 & 3 & \infty & 7 \\ 8 & 0 & 2 & \infty \\ 5 & \infty & 0 & 1 \\ 2 & \infty & \infty & 0 \end{pmatrix}
$$

**$k = 1$** (routes allowed through vertex 1): $d(2,4) \leftarrow 8 + 7 = 15$, $d(3,2) \leftarrow 5 + 3 = 8$, $d(4,2) \leftarrow 2 + 3 = 5$.

$$
D^{(1)} = \begin{pmatrix} 0 & 3 & \infty & 7 \\ 8 & 0 & 2 & 15 \\ 5 & 8 & 0 & 1 \\ 2 & 5 & \infty & 0 \end{pmatrix}
$$

**$k = 2$**: $d(1,3) \leftarrow 3 + 2 = 5$, $d(4,3) \leftarrow 5 + 2 = 7$.

$$
D^{(2)} = \begin{pmatrix} 0 & 3 & 5 & 7 \\ 8 & 0 & 2 & 15 \\ 5 & 8 & 0 & 1 \\ 2 & 5 & 7 & 0 \end{pmatrix}
$$

**$k = 3$**: $d(1,4) \leftarrow 5 + 1 = 6$, $d(2,1) \leftarrow 2 + 5 = 7$, $d(2,4) \leftarrow 2 + 1 = 3$.

$$
D^{(3)} = \begin{pmatrix} 0 & 3 & 5 & 6 \\ 7 & 0 & 2 & 3 \\ 5 & 8 & 0 & 1 \\ 2 & 5 & 7 & 0 \end{pmatrix}
$$

**$k = 4$**: $d(2,1) \leftarrow 3 + 2 = 5$, $d(3,1) \leftarrow 1 + 2 = 3$, $d(3,2) \leftarrow 1 + 5 = 6$.

$$
\boxed{D^{(4)} = \begin{pmatrix} 0 & 3 & 5 & 6 \\ 5 & 0 & 2 & 3 \\ 3 & 6 & 0 & 1 \\ 2 & 5 & 7 & 0 \end{pmatrix}}
$$

Spot check: $\delta(3,2) = 6$ realized by $3 \to 4 \to 1 \to 2$ with $1 + 2 + 3 = 6$ ✓; $\delta(4,3) = 7$ via $4 \to 1 \to 2 \to 3$ with $2 + 3 + 2 = 7$ ✓. All diagonal entries stay $0$, so there is no negative cycle.

*Key takeaway:* Each round $k$ *unlocks* one more permitted waypoint; the diagonal is the built-in negative-cycle alarm.

### Problem L1.4: DAG Shortest Paths with a Negative Arc

The DAG on $\lbrace 1,2,3,4,5 \rbrace$ (already topologically ordered) has arcs $1 \to 2$ of weight $5$, $1 \to 3$ of weight $3$, $2 \to 3$ of weight $2$, $2 \to 4$ of weight $6$, $3 \to 4$ of weight $7$, $3 \to 5$ of weight $4$, $4 \to 5$ of weight $-1$. Compute all distances from vertex $1$ in a single sweep.

**Solution**

Process vertices in topological order, relaxing each vertex's outgoing arcs once. Labels start at $d(1) = 0$, others $+\infty$.

| Vertex processed | Relaxations | Labels after |
|---|---|---|
| $1$ | $d(2) \leftarrow 5$, $d(3) \leftarrow 3$ | $d = (0, 5, 3, \infty, \infty)$ |
| $2$ | $5 + 2 = 7$ not $\lt 3$; $d(4) \leftarrow 5 + 6 = 11$ | $d = (0, 5, 3, 11, \infty)$ |
| $3$ | $3 + 7 = 10 \lt 11 \Rightarrow d(4) = 10$; $d(5) \leftarrow 3 + 4 = 7$ | $d = (0,5,3,10,7)$ |
| $4$ | $10 - 1 = 9$, not $\lt 7$ | unchanged |
| $5$ | no outgoing arcs | final |

$$
\boxed{\delta(1, \cdot) = (0, 5, 3, 10, 7); \ \text{optimal route to } 5 \text{ is } 1 \to 3 \to 5}
$$

The negative arc $4 \to 5$ causes no trouble: acyclicity forbids negative cycles, and topological order relaxes the arcs of every shortest path in order, so one sweep in $O(n + m)$ suffices.

*Key takeaway:* On a DAG, order — not weight sign — is what matters; negating weights turns the same sweep into a *longest-path* (critical-path) computation.

### Problem L1.5: The Triangle Inequality and the Predecessor Tree

Prove (a) $\delta(s,v) \le \delta(s,u) + w(u,v)$ for every arc $(u,v)$, and (b) once labels are exact, the predecessor pointers $\pi$ form a tree rooted at $s$ spanning the reachable vertices.

**Solution**

**(a)** If $\delta(s,u) = \infty$ the inequality is vacuous. Otherwise take a shortest $s$–$u$ walk $W$ of weight $\delta(s,u)$; appending the arc $(u,v)$ gives an $s$–$v$ walk of weight $\delta(s,u) + w(u,v)$. Since $\delta(s,v)$ is the minimum over *all* $s$–$v$ walks,

$$
\delta(s,v) \le \delta(s,u) + w(u,v)
$$

This is exactly the feasibility half of the Bellman system, and it is what every correct algorithm must satisfy at termination.

**(b)** Every reachable $v \neq s$ has $\pi(v) = u$ with $\delta(s,v) = \delta(s,u) + w(u,v)$ (the arc that last relaxed $v$ successfully; exactness makes it *tight*). Suppose the pointers contained a cycle $v_1 \to v_2 \to \dots \to v_r \to v_1$ (following $\pi$ backwards). Summing the $r$ tight equations and cancelling the identical multiset of $\delta$ values on both sides gives $w(C) = 0$ for that cycle. Such a zero cycle can be broken without changing any distance (delete one of its pointers and re-attach along a tight arc from outside the cycle, which exists because $s$ is reachable), so the pointer structure may always be chosen acyclic. With $n_r - 1$ arcs on $n_r$ reachable vertices and every vertex having a directed route to $s$, the result is a spanning **arborescence** rooted at $s$.

$$
\boxed{\delta(s,v) \le \delta(s,u) + w(u,v) \ \forall (u,v) \in E; \quad \pi \text{ induces a shortest-path tree}}
$$

*Key takeaway:* Feasibility ($d(v) \le d(u) + w$) plus tightness ($d(v) = d(\pi(v)) + w$) is a complete optimality certificate — verifiable in $O(m)$ without rerunning any algorithm.

### Problem L1.6: Dijkstra Fails on the Graph of Problem L1.2

Run Dijkstra from $s$ on the digraph of Problem L1.2 ($s \to a$ of weight $4$, $s \to b$ of weight $5$, $a \to c$ of weight $3$, $b \to a$ of weight $-3$, $b \to c$ of weight $6$) and identify exactly which step of the correctness proof fails.

**Solution**

**Dijkstra's trace.**

1. Extract $s$ ($d = 0$); relax to get $d(a) = 4$, $d(b) = 5$.
2. Extract $a$ ($d = 4$, the minimum key) — $a$ is now **settled**; relax $a \to c$ to get $d(c) = 7$.
3. Extract $b$ ($d = 5$); relax $b \to a$: $5 - 3 = 2 \lt 4$, but $a$ is closed, so the improvement is discarded (or, if reopening is allowed, $a$ must be re-expanded); relax $b \to c$: $11$, no improvement.
4. Extract $c$ ($d = 7$).

Dijkstra reports $d(a) = 4$, $d(c) = 7$, whereas the true values from Problem L1.2 are $\delta(s,a) = 2$ and $\delta(s,c) = 5$.

**Which proof step breaks.** In Theorem D we argued: let $y$ be the first vertex of the optimal path outside the settled set $S$; then $d(y) = \delta(s,y)$ and, crucially,

$$
\delta(s,y) \le \delta(s,u) \qquad (\ast)
$$

because the remaining segment $P_{yu}$ has nonnegative weight. Here $u = a$, $y = b$, and $(\ast)$ says $\delta(s,b) = 5 \le \delta(s,a) = 2$ — **false**, because the segment $b \to a$ has weight $-3 \lt 0$. Everything else in the proof survives; nonnegativity is used at precisely this one inequality.

$$
\boxed{\text{Dijkstra outputs } d(a) = 4 \neq 2 = \delta(s,a); \text{ the failing step is } \delta(s,y) \le \delta(s,u)}
$$

*Key takeaway:* The correct remedies are Bellman–Ford ($O(nm)$) or Johnson reweighting (Problem L3.1) — not ad-hoc reopening, which can blow up exponentially.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: Viterbi Decoding as a DAG Shortest Path

A two-state HMM (states $A$, $B$) is unrolled over three time steps. Using negative log-probabilities as arc weights: $\text{Start} \to A_1 = 0.36$, $\text{Start} \to B_1 = 1.20$; $A_1 \to A_2 = 0.22$, $A_1 \to B_2 = 1.61$, $B_1 \to A_2 = 0.92$, $B_1 \to B_2 = 0.51$; $A_2 \to A_3 = 0.36$, $A_2 \to B_3 = 1.20$, $B_2 \to A_3 = 0.69$, $B_2 \to B_3 = 0.69$. Find the most probable state sequence and its probability.

**Solution**

The trellis is a DAG in the natural time order, so one topological sweep (Theorem H) — which *is* the Viterbi recursion — suffices.

| Node | Candidates | $d$ | Backpointer |
|---|---|---|---|
| $A_1$ | $0.36$ | $0.36$ | Start |
| $B_1$ | $1.20$ | $1.20$ | Start |
| $A_2$ | $0.36 + 0.22 = 0.58$; $1.20 + 0.92 = 2.12$ | $0.58$ | $A_1$ |
| $B_2$ | $0.36 + 1.61 = 1.97$; $1.20 + 0.51 = 1.71$ | $1.71$ | $B_1$ |
| $A_3$ | $0.58 + 0.36 = 0.94$; $1.71 + 0.69 = 2.40$ | $0.94$ | $A_2$ |
| $B_3$ | $0.58 + 1.20 = 1.78$; $1.71 + 0.69 = 2.40$ | $1.78$ | $A_2$ |

The cheapest terminal node is $A_3$ with $0.94$; tracing backpointers gives $A_1 \to A_2 \to A_3$. Converting back from negative log-probability:

$$
P^{\ast} = e^{-0.94} \approx 0.391
$$

$$
\boxed{\text{Viterbi path } (A, A, A), \quad -\log P^{\ast} = 0.94, \quad P^{\ast} \approx 0.39}
$$

*Key takeaway:* Maximizing $\prod p$ equals minimizing $\sum (-\log p)$; the max-product dynamic program is a min-sum shortest path on a layered DAG, which is why Viterbi, CTC decoding and DTW share one recurrence.

### Problem L2.2: Value Iteration Is Bellman–Ford

A deterministic MDP has states $s_1, s_2, s_3$ and goal $g$ with action costs $s_1 \to s_2 = 1$, $s_2 \to s_3 = 1$, $s_3 \to g = 1$, $s_1 \to g = 10$, $s_2 \to g = 4$. Run undiscounted value iteration with $V_0(g) = 0$ and $V_0 = +\infty$ elsewhere, and explain why the iteration count matches Bellman–Ford's bound.

**Solution**

The Bellman optimality backup for a deterministic cost-minimizing MDP is

$$
V_{k+1}(s) = \min_{a} \lbrace c(s,a) + V_k(s') \rbrace
$$

which is exactly one Bellman–Ford relaxation pass on the reversed graph with target $g$.

| $k$ | $V_k(s_1)$ | $V_k(s_2)$ | $V_k(s_3)$ |
|---|---|---|---|
| $0$ | $\infty$ | $\infty$ | $\infty$ |
| $1$ | $10$ | $4$ | $1$ |
| $2$ | $\min(10, 1 + 4) = 5$ | $\min(4, 1 + 1) = 2$ | $1$ |
| $3$ | $\min(10, 1 + 2) = 3$ | $2$ | $1$ |
| $4$ | $3$ | $2$ | $1$ (fixed point) |

The optimal policy is "always step to the next state": $s_1 \to s_2 \to s_3 \to g$ with cost $3$.

**Why three iterations.** $V_k$ is exact for all states whose optimal plan uses at most $k$ actions — the same hop-count induction as Theorem E. The optimal plan from $s_1$ has $3$ steps, so convergence occurs at $k = 3$ and iteration $4$ merely confirms the fixed point.

$$
\boxed{V^{\ast} = (3, 2, 1) \text{ for } (s_1, s_2, s_3); \ \text{converged in } 3 = \text{optimal hop count iterations}}
$$

*Key takeaway:* Value iteration is Bellman–Ford on a stochastic graph. In the discounted case the factor $\gamma \lt 1$ makes the backup a $\gamma$-contraction, replacing the "no negative cycle" hypothesis with a geometric convergence rate $\gamma^k$.

### Problem L2.3: A$^{\ast}$ on a Grid — Consistency and Expansion Count

On the grid $\lbrace 0,1,2,3,4 \rbrace \times \lbrace 0,1,2 \rbrace$ with unit-cost 4-neighbour moves, start $S = (2,1)$ and target $T = (4,1)$, use the Manhattan heuristic $h(x,y) = \vert x - 4 \vert + \vert y - 1 \vert$. (a) Verify consistency. (b) Count the vertices expanded by A$^{\ast}$ versus Dijkstra.

**Solution**

**(a) Consistency.** A unit move changes exactly one coordinate by one, so $\vert h(u) - h(v) \vert \le 1 = w(u,v)$ for adjacent cells, giving $h(u) \le w(u,v) + h(v)$; and $h(T) = 0$. Equivalently the reduced costs are

$$
\hat{w}(u,v) = 1 + h(v) - h(u) \in \lbrace 0, 2 \rbrace \ge 0
$$

with value $0$ for a move that reduces Manhattan distance and $2$ for a move that increases it — the heuristic makes progress free and detours doubly expensive.

**(b) Expansions.** $\delta(S,T) = 2$, and A$^{\ast}$ expands only vertices with $f(v) = d(v) + h(v) \le 2$. Those are exactly the cells on the straight corridor:

$$
(2,1) \ [f = 0 + 2], \quad (3,1) \ [f = 1 + 1], \quad (4,1) \ [f = 2 + 0] \quad \Rightarrow \quad 3 \text{ expansions}
$$

Any cell off the corridor has $f \ge 4$ — e.g. $(2,0)$ has $d = 1$, $h = 3$, $f = 4$ — and is never expanded.

Dijkstra ($h \equiv 0$) expands every cell with $d \le 2$: one cell at distance $0$, four at distance $1$ — $(1,1), (3,1), (2,0), (2,2)$ — and six at distance $2$ — $(0,1), (4,1), (1,0), (1,2), (3,0), (3,2)$:

$$
1 + 4 + 6 = 11 \text{ expansions}
$$

$$
\boxed{h \text{ is consistent}; \ \text{A}^{\ast}: 3 \text{ expansions vs. Dijkstra}: 11}
$$

*Key takeaway:* A consistent heuristic does not change *what* is optimal, only *how much of the graph must be touched* — the reduced-cost view makes the pruning quantitative: cells are skipped exactly when $f \gt \delta(S,T)$.

### Problem L2.4: Currency Arbitrage as a Negative Cycle

Exchange rates are $r(\text{USD} \to \text{EUR}) = 0.90$, $r(\text{EUR} \to \text{GBP}) = 0.85$, $r(\text{GBP} \to \text{USD}) = 1.35$. Show how to detect arbitrage with a shortest-path algorithm, and determine whether this cycle is profitable.

**Solution**

Arbitrage means a cycle with $\prod_i r_i \gt 1$. Taking logarithms converts the product into a sum; setting

$$
w(u,v) = -\ln r(u \to v)
$$

turns "$\prod r_i \gt 1$" into "$\sum w_i \lt 0$" — a **negative cycle**, detectable by Bellman–Ford in $O(nm)$.

Numerically:

$$
w_1 = -\ln 0.90 = 0.10536, \quad w_2 = -\ln 0.85 = 0.16252, \quad w_3 = -\ln 1.35 = -0.30010
$$

$$
w(C) = 0.10536 + 0.16252 - 0.30010 = -0.03222 \lt 0
$$

So the cycle is a negative cycle and the loop is profitable. The gain factor is

$$
\prod_i r_i = e^{-w(C)} = e^{0.03222} \approx 1.0327
$$

i.e. about $3.3\%$ per round trip (directly: $0.90 \times 0.85 \times 1.35 = 1.03275$).

$$
\boxed{w(C) = -0.0322 \lt 0 \Rightarrow \text{arbitrage of } \approx 3.27\% \text{ per cycle}}
$$

*Key takeaway:* The logarithm is the standard bridge between multiplicative and additive objectives; it is the same transformation that converts maximum-likelihood decoding into shortest paths (Problem L2.1).

### Problem L2.5: Shortest-Path Tree Versus Minimum Spanning Tree

A hub $s$ connects to $a$, $b$, $c$ with weight $10$ each, and the rim edges $ab$ and $bc$ have weight $1$. Compute the MST, the shortest-path tree from $s$, their total weights, and the worst distance *stretch* incurred by routing on the MST.

**Solution**

**MST** (Kruskal on weights $1, 1, 10, 10, 10$): take $ab$ and $bc$ (weight $1$ each), then any one spoke, say $sa$:

$$
w(\mathrm{MST}) = 1 + 1 + 10 = 12
$$

**Shortest-path tree from $s$.** Each rim vertex is reachable directly at cost $10$, while a detour costs at least $10 + 1 = 11$, so $\delta(s,a) = \delta(s,b) = \delta(s,c) = 10$ and the SPT is the three spokes:

$$
w(\mathrm{SPT}) = 30
$$

**Stretch.** Routing inside the MST, the distance from $s$ to $c$ is $10 + 1 + 1 = 12$ against a true distance of $10$:

$$
\text{stretch} = \frac{12}{10} = 1.2
$$

$$
\boxed{w(\mathrm{MST}) = 12 \lt w(\mathrm{SPT}) = 30, \ \text{but MST routing stretches } s \to c \text{ by } 1.2\times}
$$

Generalizing the construction (a long rim of cheap edges hanging off one expensive spoke) pushes the stretch to $\Theta(n)$: an MST can be an arbitrarily bad router.

*Key takeaway:* Minimizing total *construction* cost and minimizing *travel* cost are different objectives; sparsification in graph ML faces the same trade-off between preserving connectivity and preserving distances (spanners interpolate between the two).

### Problem L2.6: 0–1 BFS for Two-Tier Networks

A network has free links (weight $0$) and metered links (weight $1$): $s \to a$ free, $s \to b$ metered, $a \to c$ metered, $b \to c$ free, $c \to t$ free, $b \to t$ metered. Compute distances with the deque-based 0–1 BFS and state why it beats Dijkstra asymptotically.

**Solution**

**0–1 BFS** replaces the heap by a double-ended queue: on relaxing an edge of weight $0$ push the endpoint to the **front**, on weight $1$ push it to the **back**. The deque then always holds at most two distinct label values, which preserves Dijkstra's extraction order without any comparisons.

| Pop | Relaxations | Deque afterwards |
|---|---|---|
| $s$ $(0)$ | $a \leftarrow 0$ (front), $b \leftarrow 1$ (back) | $[a(0), b(1)]$ |
| $a$ $(0)$ | $c \leftarrow 0 + 1 = 1$ (back) | $[b(1), c(1)]$ |
| $b$ $(1)$ | $b \to c$: $1 + 0 = 1$, no gain; $t \leftarrow 2$ (back) | $[c(1), t(2)]$ |
| $c$ $(1)$ | $t \leftarrow 1 + 0 = 1$ (front) | $[t(1), t(2)]$ |
| $t$ $(1)$ | settle; the stale entry $t(2)$ is discarded on pop | $[\,]$ |

$$
\boxed{\delta(s,\cdot) = (0, 0, 1, 1, 1) \text{ for } (s,a,b,c,t); \ \text{cost } O(n + m) \text{ vs. Dijkstra's } O(m \log n)}
$$

**Why it is correct.** With weights in $\lbrace 0, 1 \rbrace$ all labels currently in the deque lie in $\lbrace D, D+1 \rbrace$ for the current front value $D$; pushing zero-weight successors to the front and unit-weight successors to the back maintains that sorted invariant, which is precisely what a priority queue would enforce — for free.

*Key takeaway:* Structure in the weights buys asymptotics: $\lbrace 0,1 \rbrace$ gives a deque, small integers give Dial's buckets in $O(m + nC)$, and integer word-RAM weights give Thorup's $O(m)$ undirected algorithm.

## Level 3 — Challenge

### Problem L3.1: Johnson's Reweighting

Let $p : V \to \mathbb{R}$ satisfy $p(v) \le p(u) + w(u,v)$ for every arc, and define $\hat{w}(u,v) = w(u,v) + p(u) - p(v)$. (a) Prove $\hat{w} \ge 0$. (b) Prove shortest paths are unchanged and give the distance-recovery formula. (c) Apply it to the graph of Problem L1.2 and recover the distances with Dijkstra.

**Solution**

**(a)** The hypothesis rearranges to $w(u,v) + p(u) - p(v) \ge 0$, i.e. $\hat{w}(u,v) \ge 0$. Such potentials always exist when no negative cycle does: add an artificial source $q$ with zero-weight arcs to every vertex and set $p(v) = \delta(q,v)$, which satisfies the inequality by the triangle inequality of Problem L1.5.

**(b)** For any $u$–$v$ path $P = \langle u = x_0, \dots, x_k = v \rangle$ the potential terms telescope:

$$
\hat{w}(P) = \sum_{i=1}^{k} \big( w(x_{i-1},x_i) + p(x_{i-1}) - p(x_i) \big) = w(P) + p(u) - p(v)
$$

The correction $p(u) - p(v)$ depends only on the endpoints, so all $u$–$v$ paths are shifted by the same constant and their ranking is preserved: a path is shortest under $\hat{w}$ iff it is shortest under $w$. Taking minima,

$$
\boxed{\delta(u,v) = \hat{\delta}(u,v) - p(u) + p(v)}
$$

**(c)** For the graph $s \to a$ of weight $4$, $s \to b$ of weight $5$, $a \to c$ of weight $3$, $b \to a$ of weight $-3$, $b \to c$ of weight $6$, one Bellman–Ford run from an artificial source gives $p = (p_s, p_a, p_b, p_c) = (0, -3, 0, 0)$. Reduced weights:

| Arc | $w$ | $\hat{w} = w + p(u) - p(v)$ |
|---|---|---|
| $s \to a$ | $4$ | $4 + 0 + 3 = 7$ |
| $s \to b$ | $5$ | $5 + 0 - 0 = 5$ |
| $b \to a$ | $-3$ | $-3 + 0 + 3 = 0$ |
| $a \to c$ | $3$ | $3 - 3 - 0 = 0$ |
| $b \to c$ | $6$ | $6 + 0 - 0 = 6$ |

All nonnegative, so Dijkstra applies: $\hat{\delta}(s, \cdot) = (0, 5, 5, 5)$ for $(s,a,b,c)$ — via $s \to b$ then the two zero-cost arcs. Recovering with $\delta(s,v) = \hat{\delta}(s,v) + p(v) - p(s)$:

$$
\delta(s,a) = 5 - 3 = 2, \qquad \delta(s,b) = 5, \qquad \delta(s,c) = 5
$$

matching Problem L1.2 exactly.

**Cost.** One Bellman–Ford ($O(nm)$) plus $n$ Dijkstra runs gives all-pairs distances in $O(nm + n^2\log n)$, beating $\Theta(n^3)$ whenever $m = o(n^2 / \log n)$.

*Key takeaway:* Potentials are the dual variables of the shortest-path LP; reweighting by them is the same device that powers A$^{\ast}$ (Problem L3.2) and min-cost-flow algorithms (Topic 05).

### Problem L3.2: Consistency, Admissibility, and Node Reopening

(a) Prove that a consistent heuristic is admissible. (b) Prove that with a consistent heuristic A$^{\ast}$ never reopens a closed vertex. (c) Exhibit an *admissible but inconsistent* heuristic that forces a reopening.

**Solution**

**(a) Consistency $\Rightarrow$ admissibility.** Let $P = \langle v = x_0, x_1, \dots, x_k = t \rangle$ be a shortest $v$–$t$ path. Applying $h(x_{i-1}) \le w(x_{i-1},x_i) + h(x_i)$ for $i = 1, \dots, k$ and telescoping:

$$
h(v) \le \sum_{i=1}^{k} w(x_{i-1},x_i) + h(t) = \delta(v,t)
$$

since $h(t) = 0$. (If $t$ is unreachable, $\delta(v,t) = \infty$ and the bound is trivial.)

**(b) No reopening.** By Theorem G, A$^{\ast}$ with key $f(v) = d(v) + h(v)$ is Dijkstra run on $\hat{w}(u,v) = w(u,v) + h(v) - h(u) \ge 0$, since the keys differ from Dijkstra's by the constant $h(s)$:

$$
\hat{d}(v) = d(v) + h(v) - h(s) = f(v) - h(s)
$$

Dijkstra's invariant (Theorem D) says a vertex extracted from the queue already carries its exact $\hat{\delta}$ value; no later relaxation can lower an exact label, so no closed vertex is ever improved. $\blacksquare$

**(c) A concrete reopening.** Vertices $S, A, B, G$ with arcs $S \to A = 5$, $S \to B = 1$, $B \to A = 1$, $A \to G = 10$. True distances to $G$: $\delta(A,G) = 10$, $\delta(B,G) = 11$, $\delta(S,G) = 12$. Choose

$$
h(G) = 0, \quad h(A) = 0, \quad h(B) = 11, \quad h(S) = 0
$$

Every value satisfies $h \le \delta(\cdot, G)$, so $h$ is **admissible**. But consistency fails on $(B,A)$: $h(B) = 11 \le w(B,A) + h(A) = 1$ is false, i.e. $\hat{w}(B,A) = 1 + 0 - 11 = -10 \lt 0$.

Trace: expand $S$ ($f = 0$), generating $A$ with $g = 5$, $f = 5$ and $B$ with $g = 1$, $f = 12$. Expand $A$ ($f = 5$) — **closed with the suboptimal $g = 5$** — generating $G$ with $g = 15$. Expand $B$ ($f = 12$), which relaxes $A$ to $g = 1 + 1 = 2$: the closed vertex $A$ must be **reopened**, re-expanded, and $G$ corrected to $g = 12$, which is optimal.

$$
\boxed{\text{consistency} \Rightarrow \text{admissibility and zero reopenings}; \text{ admissibility alone permits re-expansion}}
$$

*Key takeaway:* Inconsistent heuristics can cost exponentially many re-expansions in the worst case; the standard repair is to run A$^{\ast}$ with the *pathmax* correction $h(v) \leftarrow \max(h(v), h(u) - w(u,v))$, which restores consistency along explored arcs.

### Problem L3.3: Adding a Constant to Every Weight

Let $w_c(e) = w(e) + c$ for a constant $c \gt 0$. (a) Show by example that shortest paths are not preserved. (b) Characterize exactly when they are. (c) Explain why this rules out "shift the weights up" as a fix for negative edges under Dijkstra.

**Solution**

**(a) Counterexample.** Take $s \to t$ of weight $5$, and the route $s \to a \to b \to t$ with all three arcs of weight $1$. Originally the three-hop route wins with $3 \lt 5$. Add $c = 2$ to every arc:

$$
w_c(\text{direct}) = 7, \qquad w_c(\text{three-hop}) = 3 + 3 \cdot 2 = 9
$$

Now the direct arc wins. The optimum flipped.

**(b) Characterization.** For a path $P$ with $\vert P \vert$ arcs,

$$
w_c(P) = w(P) + c \cdot \vert P \vert
$$

so the shift is a **hop-count penalty**, not a uniform offset. Comparing two $s$–$t$ paths $P, Q$:

$$
w_c(P) - w_c(Q) = \big( w(P) - w(Q) \big) + c \big( \vert P \vert - \vert Q \vert \big)
$$

The ordering of *all* $s$–$t$ path pairs is preserved for every $c \gt 0$ iff every pair of $s$–$t$ paths compared has equal hop count — which holds, for instance, in a **layered DAG** (all $s$–$t$ paths have the same length, as in an HMM trellis or a fixed-depth neural architecture). More generally, the optimum is unchanged for a *specific* $c$ iff the optimal path minimizes $w(P) + c \vert P \vert$, i.e. iff it survives the lexicographic tie-break induced by that penalty.

$$
\boxed{w_c(P) = w(P) + c \vert P \vert; \ \text{order preserved} \iff \text{competing paths have equal hop counts}}
$$

**(c) Consequence for negative weights.** Shifting by $c = \max_e \vert w(e) \vert$ makes all weights nonnegative, so Dijkstra *runs* — but it then solves a different problem, biased against long paths, and returns wrong answers on any graph where the optimum uses more hops than a rival. The correct fix is Johnson's **path-dependent** reweighting $\hat{w} = w + p(u) - p(v)$ (Problem L3.1), whose correction telescopes to a constant per endpoint pair rather than growing with hop count.

*Key takeaway:* Only reweightings of the *potential* form $p(u) - p(v)$ preserve shortest paths; constants do not, because they are not telescoping.

### Problem L3.4: Shortest Paths in the $(\min, +)$ Tropical Semiring

Let $W$ be the weight matrix ($W_{ii} = 0$, $W_{ij} = w(i,j)$ or $+\infty$) and define tropical matrix multiplication $(A \otimes B)_{ij} = \min_k \lbrace A_{ik} + B_{kj} \rbrace$. (a) Prove that $(W^{\otimes k})_{ij}$ is the minimum weight of an $i$–$j$ walk with at most $k$ arcs. (b) Deduce $W^{\otimes (n-1)} = \delta$ and give the repeated-squaring complexity. (c) Verify on the three-vertex graph with arcs $1 \to 2 = 3$, $1 \to 3 = 8$, $2 \to 3 = 2$, $3 \to 1 = 4$.

**Solution**

**(a) Induction on $k$.** For $k = 1$, $W_{ij}$ is the best walk with at most one arc ($0$ when $i = j$ thanks to the diagonal). Assume the claim for $k-1$. Any $i$–$j$ walk with at most $k$ arcs either has at most $k-1$ arcs, or decomposes as an $i$–$\ell$ walk with at most $k-1$ arcs followed by the arc $(\ell, j)$. Minimizing over the split point $\ell$ — and noting $W_{jj} = 0$ absorbs the shorter case — gives

$$
(W^{\otimes k})_{ij} = \min_{\ell} \lbrace (W^{\otimes (k-1)})_{i\ell} + W_{\ell j} \rbrace
$$

which is exactly the tropical product. $\blacksquare$

**(b)** Absent negative cycles, an optimal walk may be taken simple, hence uses at most $n-1$ arcs, so

$$
\boxed{W^{\otimes (n-1)} = \delta, \quad \text{and } W^{\otimes k} = W^{\otimes (n-1)} \text{ for all } k \ge n-1}
$$

Because $\otimes$ is associative, repeated squaring reaches exponent $\ge n-1$ in $\lceil \log_2 (n-1) \rceil$ products, each costing $\Theta(n^3)$:

$$
T = \Theta(n^3 \log n)
$$

which is worse than Floyd–Warshall's $\Theta(n^3)$ — the reason the semiring view is prized for *insight* (and for min-plus algebra in tropical geometry) rather than for speed. Note also that Strassen-style fast matrix multiplication does **not** transfer, since $(\min, +)$ has no additive inverses; whether truly subcubic min-plus multiplication exists is the APSP conjecture.

**(c) Verification.**

$$
W = \begin{pmatrix} 0 & 3 & 8 \\ \infty & 0 & 2 \\ 4 & \infty & 0 \end{pmatrix}, \qquad W^{\otimes 2} = \begin{pmatrix} 0 & 3 & 5 \\ 6 & 0 & 2 \\ 4 & 7 & 0 \end{pmatrix}
$$

Sample entries: $(W^{\otimes 2})_{13} = \min(0 + 8, \ 3 + 2, \ 8 + 0) = 5$ via $1 \to 2 \to 3$; $(W^{\otimes 2})_{21} = \min(\infty, \ \infty, \ 2 + 4) = 6$ via $2 \to 3 \to 1$; $(W^{\otimes 2})_{32} = \min(4 + 3, \ \infty, \ \infty) = 7$ via $3 \to 1 \to 2$. Since $n - 1 = 2$, this matrix is already $\delta$, and indeed $W^{\otimes 3} = W^{\otimes 2}$.

*Key takeaway:* Replacing $(+, \times)$ by $(\min, +)$ turns linear algebra into shortest paths, $(\max, \times)$ into Viterbi, and $(+, \times)$ into the forward algorithm — one algebraic template, three classic dynamic programs.